# 02c FedUP under FuGuard's protocol (not in the FedUP paper)
FuGuard's exact setup on FuGuard's four datasets (SVHN, EuroSAT, CIFAR-10, CIFAR-100): 10 clients, **client 3 backdoored** (4x4 patch -> class 9, FuGuard's trigger constants), 3-block ConvNet, Adam, 49 FedAvg rounds, all data used.

**Question:** does a filter trained on clean centroids on top of a frozen extractor remove a backdoor, and how does it compare with FuGuard? The comparison with the FuGuard paper's numbers is in `03_analysis/03c_compare_with_papers.ipynb`.

SVHN and EuroSAT need their datasets first (`scripts/download_datasets.py --only svhn eurosat`). FedUP alone is enough for the comparison; add `retrain eraseclient` to `--methods` for the baselines.

In [ ]:
# find the repo root (works from notebooks/<group>/, from the repo root, on Kaggle and on Colab) and set up the folders
import os, sys, shutil
from pathlib import Path
KAGGLE_REPO = Path("/kaggle/input/datasets/sakhiurrahman/fedupds/FedUP_Reproduction")   # your Kaggle dataset (edit if you move it)
def _find_repo():
    for p in [Path.cwd(), *Path.cwd().parents, Path("/kaggle/working/FedUP_Reproduction"), Path("/content/FedUP_Reproduction"), KAGGLE_REPO]:
        if (p / "fedup_repro").exists():
            return p
    kin = Path("/kaggle/input")
    if kin.exists():                                    # any other layout of the input dataset, up to 6 levels deep
        for dp, dn, _ in os.walk(kin):
            if "fedup_repro" in dn:
                return Path(dp)
            if len(Path(dp).relative_to(kin).parts) > 6:
                dn[:] = []
    return None
ROOT = _find_repo()
assert ROOT is not None, "FedUP_Reproduction folder not found: upload it (Kaggle: as a Dataset) or cd into it"
if str(ROOT).startswith("/kaggle/input"):              # input datasets are read-only: work on a copy of the code ...
    SRC, ROOT = ROOT, Path("/kaggle/working") / ROOT.name
    if not ROOT.exists():
        shutil.copytree(SRC, ROOT, ignore=lambda d, names: [n for n in names if n == "raw" and Path(d).name == "data"])
    os.environ["FEDUP_BUNDLED_RAW"] = str(SRC / "data" / "raw")      # ... and LINK (not copy) the datasets that ship with it
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
from fedup_repro import colab
PERSIST_ROOT = None        # Colab + Drive: "/content/drive/MyDrive/fedup_store" keeps data/ and artifacts/ across sessions (mount Drive first)
print("platform:", colab.setup_dirs(PERSIST_ROOT), "| repo:", ROOT)

import subprocess, sys
def run(*args):
    subprocess.run([sys.executable, *args], check=True)      # multi-GPU is automatic (one worker per visible GPU)

In [ ]:
run('scripts/download_datasets.py', '--only', 'svhn', 'eurosat', 'cifar10', 'cifar100')

In [ ]:
run('scripts/run_experiment.py', 'cross', '--methods', 'fedup', '--seeds', '0', '1')

In [ ]:
run('scripts/make_reports.py')
from fedup_repro import aggregate as A, comparison as C
display(A.table_cross_protocol('cross'))
display(C.fuguard_vs_fedup_compact())